### Preparing dataset and csv files for training and validation purposes

In [2]:
import csv
import os
import torchaudio

In [13]:
drums = "/home/lois/wavenext/data/bach-violin-dataset"
with open("data/bach_violin_dataset.csv", "w", newline="") as csvfile:
    for root, dirs, files in os.walk(drums):
        for file in files:
            if file.endswith(".wav"):
                full_path = os.path.join(root, file)
                writer = csv.writer(csvfile)
                writer.writerow([full_path])
                print(full_path)

/home/lois/wavenext/data/bach-violin-dataset/bach-violin/audio/oliver-colbentson/oliver-colbentson_bwv1006_mov5.wav
/home/lois/wavenext/data/bach-violin-dataset/bach-violin/audio/oliver-colbentson/oliver-colbentson_bwv1006_mov7.wav
/home/lois/wavenext/data/bach-violin-dataset/bach-violin/audio/oliver-colbentson/oliver-colbentson_bwv1006_mov1.wav
/home/lois/wavenext/data/bach-violin-dataset/bach-violin/audio/oliver-colbentson/oliver-colbentson_bwv1006_mov6.wav
/home/lois/wavenext/data/bach-violin-dataset/bach-violin/audio/oliver-colbentson/oliver-colbentson_bwv1006_mov3.wav
/home/lois/wavenext/data/bach-violin-dataset/bach-violin/audio/oliver-colbentson/oliver-colbentson_bwv1006_mov4.wav
/home/lois/wavenext/data/bach-violin-dataset/bach-violin/audio/misc/silei-li_bwv1003.wav
/home/lois/wavenext/data/bach-violin-dataset/bach-violin/audio/misc/kinga-augustyn_bwv1005_mov2.wav
/home/lois/wavenext/data/bach-violin-dataset/bach-violin/audio/isabella-stewart-gardner/karen-gomyo_bwv1006.wav
/ho

In [5]:
# Check the content of the CSV file, audio duration, sample rate, etc.

duration = []

with open("data/taiko_dataset.csv", "r") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        print(row)
        audio, sr = torchaudio.load(row[0])
        audio_duration = audio.shape[1] / sr
        duration.append(audio_duration)
        print(f"Audio duration: {audio_duration:.2f} seconds, Sample rate: {sr} Hz")

print(f"Average audio duration: {sum(duration)/len(duration):.2f} seconds")
print(f"Minimum audio duration: {min(duration):.2f} seconds")
print(f"Maximum audio duration: {max(duration):.2f} seconds")
print(f"Total duration: {sum(duration) / 3600.}")

['/home/lois/models_benchmark/downloads/audio/52636805-2a41-4002-895c-3cd5bce39ced/ohira_taiko_solo_10.29.2024.wav']
Audio duration: 338.26 seconds, Sample rate: 48000 Hz
['/home/lois/models_benchmark/downloads/audio/52636805-2a41-4002-895c-3cd5bce39ced/shime_taiko_solo_10.29.2024.wav']
Audio duration: 390.48 seconds, Sample rate: 48000 Hz
['/home/lois/models_benchmark/downloads/audio/52636805-2a41-4002-895c-3cd5bce39ced/ojime_taiko_solo_10.29.2024.wav']
Audio duration: 460.79 seconds, Sample rate: 48000 Hz
['/home/lois/models_benchmark/downloads/audio/52636805-2a41-4002-895c-3cd5bce39ced/atari_gane_solo_10.29.2024.wav']
Audio duration: 582.45 seconds, Sample rate: 48000 Hz
['/home/lois/models_benchmark/downloads/audio/52636805-2a41-4002-895c-3cd5bce39ced/nagado_taiko_solo_10.29.2024.wav']
Audio duration: 412.90 seconds, Sample rate: 48000 Hz
['/home/lois/models_benchmark/downloads/audio/52636805-2a41-4002-895c-3cd5bce39ced/rims_solo_10.30.2024.wav']
Audio duration: 332.18 seconds, Sam

In [5]:
import random
train_set = []
val_set = []
test_set = []
with open("data/taiko_dataset.csv", "r") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        if random.random() < 0.8:
            train_set.append(row[0])
        elif random.random() < 0.2:
            test_set.append(row[0])
        else:
            val_set.append(row[0])
with open("data/taiko_train.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    for path in train_set:
        writer.writerow([path])
with open("data/taiko_val.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    for path in val_set:
        writer.writerow([path])
with open("data/taiko_test.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    for path in test_set:
        writer.writerow([path])

### Split data into chunks if needed

In [6]:
import torchaudio.functional as F
from pathlib import Path

# for latent diffusion bridges model

output_dir = 'data/diff_bridges/taiko'
chunk_lenght = 409600 #(17s at 24k)
sample_rate = 24000
with open("data/taiko_dataset.csv", "r") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        print(row)
        path = Path(str(row))
        audio, sr = torchaudio.load(row[0])
        if sr != sample_rate:
            audio = F.resample(audio, sr, sample_rate)

        if audio.shape[0] > 1:
            audio = audio.mean(dim=0, keepdim=True)
        total = audio.shape[1]
        n_chunks = (total - chunk_lenght) // chunk_lenght + 1
        for i in range(n_chunks):
            start = i * chunk_lenght
            chunk = audio[:, start : start + chunk_lenght]
            if chunk.shape[1] < chunk_lenght:
                break  # drop last incomplete chunk
            out = output_dir + f"/taiko_{path.stem}_chunk{i:05d}.wav"
            torchaudio.save(str(out), chunk.clamp(-1, 1), sample_rate)


['/home/lois/models_benchmark/downloads/audio/52636805-2a41-4002-895c-3cd5bce39ced/ohira_taiko_solo_10.29.2024.wav']
['/home/lois/models_benchmark/downloads/audio/52636805-2a41-4002-895c-3cd5bce39ced/shime_taiko_solo_10.29.2024.wav']
['/home/lois/models_benchmark/downloads/audio/52636805-2a41-4002-895c-3cd5bce39ced/ojime_taiko_solo_10.29.2024.wav']
['/home/lois/models_benchmark/downloads/audio/52636805-2a41-4002-895c-3cd5bce39ced/atari_gane_solo_10.29.2024.wav']
['/home/lois/models_benchmark/downloads/audio/52636805-2a41-4002-895c-3cd5bce39ced/nagado_taiko_solo_10.29.2024.wav']
['/home/lois/models_benchmark/downloads/audio/52636805-2a41-4002-895c-3cd5bce39ced/rims_solo_10.30.2024.wav']
['/home/lois/models_benchmark/downloads/audio/52636805-2a41-4002-895c-3cd5bce39ced/okedo_taiko_solo_10.29.2024.wav']
